In [0]:
###TES APAKAH KAGGLE TOKEN SUDAH CONNECT DI KEY VAULTS

dbutils.secrets.list("kv-scope")
dbutils.secrets.get("kv-scope", "kaggle-api-token")

'[REDACTED]'

In [0]:
###DOWNLOAD DATASET DARI KAGGLE (MENGGUNAKAN KEY VAULTS PADA AWAL TADI)

import os, subprocess
os.environ["KAGGLE_API_TOKEN"] = dbutils.secrets.get("kv-scope", "kaggle-api-token")
subprocess.run(["pip", "install", "-U", "kaggle"], check=True)

from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.dataset_download_files("olistbr/brazilian-ecommerce", path="/tmp/olist", unzip=True)

Looking in indexes: [REDACTED]
  Attempting uninstall: certifi
    Found existing installation: certifi 2025.11.12
    Not uninstalling certifi at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-14344817-57c6-49dc-8550-8ddb149083c6
    Can't uninstall 'certifi'. No files were found to uninstall.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [kaggle]
Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce


In [0]:
###IMPORT FILE KE VOLUME DENGAN NAMA BRONZE

import os, subprocess
from datetime import date
from pyspark.sql import functions as F

os.environ["KAGGLE_API_TOKEN"] = dbutils.secrets.get("kv-scope", "kaggle-api-token")
subprocess.run(["pip", "install", "-U", "kaggle"], check=True)
from kaggle.api.kaggle_api_extended import KaggleApi

catalog = "databrics_ecommerce"
today   = date.today().isoformat()

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.bronze")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.bronze.landing")

landing = f"/Volumes/{catalog}/bronze/landing/{today}"
os.makedirs(landing, exist_ok=True)

api = KaggleApi(); api.authenticate()
api.dataset_download_files("olistbr/brazilian-ecommerce", path=landing, unzip=True)

for f in os.listdir(landing):
    if not f.endswith(".csv"):
        continue
    name = f.replace("olist_", "").replace("_dataset.csv", "").replace(".csv", "")
    df = (spark.read.option("header", True).csv(f"{landing}/{f}")
          .withColumn("ingestion_date", F.lit(today)))
    df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.bronze.{name}")
    print(name, df.count())

Looking in indexes: [REDACTED]
Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce
product_category_name_translation 71
customers 99441
geolocation 1000163
order_items 112650
order_payments 103886
order_reviews 104162
orders 99441
products 32951
sellers 3095
